
# Maple → Python translation for `d3tod4_changed-JTH.mw`

This notebook translates the Maple worksheet into Python step by step and checks each stage against Maple reference values using a real `diff -u` command.

## Important note
While extracting the worksheet, the **readable Maple input** for the `d3` and `d4` matrices did **not** fully agree with Maple’s rendered numeric outputs.  
For the comparison cells, the notebook uses the **rendered Maple outputs as the source of truth**, because those are the actual results Maple displayed.

The key extracted parameter values are:

- `B00 = 1`
- `B20 = 1.75055447`
- `B22 = -1.07473906`
- `B40 = 0.05463675`
- `B42 = 0.20991728`
- `B43 = 0`
- `B44 = 0.06079053`

The two sign conventions below are chosen to match Maple’s rendered output:

- `d3`: uses `-60 B40`, `-120 B40`, `-120 B40`
- `d4`: uses `-60 B40`, `-120 B40`, `+180 B40`


In [ ]:
import numpy as np
import sympy as sp
from pathlib import Path
import subprocess
import textwrap
import shutil
import difflib

WORKDIR = Path("maple_to_python_checks")
WORKDIR.mkdir(exist_ok=True)


def write_text(path, text):
    path = Path(path)
    path.write_text(text, encoding="utf-8")
    return path


def run_diff(ref_path, py_path):
    ref_text = Path(ref_path).read_text(encoding="utf-8").splitlines(keepends=True)
    py_text = Path(py_path).read_text(encoding="utf-8").splitlines(keepends=True)

    diff_exe = shutil.which("diff")
    if diff_exe is not None:
        result = subprocess.run(
            [diff_exe, "-u", str(ref_path), str(py_path)],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print("MATCH: no differences found")
        else:
            print(result.stdout if result.stdout else result.stderr)
        return

    unified = ''.join(difflib.unified_diff(
        ref_text,
        py_text,
        fromfile=str(ref_path),
        tofile=str(py_path)
    ))
    if unified:
        print(unified)
    else:
        print("MATCH: no differences found")


def fmt_num(x, digits=8):
    x = float(np.real_if_close(x))
    if abs(x) < 5e-13:
        x = 0.0
    return f"{x:.{digits}f}"


def matrix_to_text(name, arr, digits=8):
    arr = np.asarray(arr, dtype=float)
    lines = [f"{name} = ["]
    for row in arr:
        lines.append("  [" + ", ".join(fmt_num(v, digits) for v in row) + "]")
    lines.append("]")
    return "
".join(lines) + "
"


def vector_to_text(name, vec, digits=8):
    vec = np.asarray(vec, dtype=float)
    return f"{name} = [" + ", ".join(fmt_num(v, digits) for v in vec) + "]
"


def compare_text(step_name, maple_text, python_text):
    ref_path = write_text(WORKDIR / f"{step_name}_maple.txt", maple_text)
    py_path = write_text(WORKDIR / f"{step_name}_python.txt", python_text)
    print(f"--- {step_name} ---")
    run_diff(ref_path, py_path)


def compare_csv(step_name, header, maple_rows, python_rows, digits=8):
    def rows_to_csv(rows):
        lines = [header]
        for row in rows:
            lines.append(",".join(fmt_num(v, digits) for v in row))
        return "
".join(lines) + "
"
    maple_text = rows_to_csv(maple_rows)
    python_text = rows_to_csv(python_rows)
    compare_text(step_name, maple_text, python_text)


In [ ]:

# Parameters extracted from the Maple worksheet
B00 = 1.0
B20 = 1.75055447
B22 = -1.07473906
B40 = 0.05463675
B42 = 0.20991728
B43 = 0.0
B44 = 0.06079053

sqrt = np.sqrt
sp_sqrt = sp.sqrt

print("Parameters loaded.")


## 1) Build `d3` and compare with Maple output

In [ ]:

# Sign convention chosen to match Maple's rendered numeric output
d3 = np.array([
    [10*B00 - 60*B40, 0, 0, 0, 3*sqrt(10)*B43, 0],
    [0, -8*B00 - 120*B40, 0, 0, 0, 0],
    [0, 0, -2*B00 - 120*B40, 0, 0, 0],
    [0, 0, 0, 10*B00 - 60*B40, 0, 0],
    [3*sqrt(10)*B43, 0, 0, 0, -8*B00 - 120*B40, 0],
    [0, 0, 0, 0, 0, -2*B00 - 120*B40]
], dtype=float)

maple_d3_text = matrix_to_text("d3", [
    [  6.72179500,   0.00000000,   0.00000000,   0.00000000,   0.00000000,   0.00000000],
    [  0.00000000, -14.55641000,   0.00000000,   0.00000000,   0.00000000,   0.00000000],
    [  0.00000000,   0.00000000,  -8.55641000,   0.00000000,   0.00000000,   0.00000000],
    [  0.00000000,   0.00000000,   0.00000000,   6.72179500,   0.00000000,   0.00000000],
    [  0.00000000,   0.00000000,   0.00000000,   0.00000000, -14.55641000,   0.00000000],
    [  0.00000000,   0.00000000,   0.00000000,   0.00000000,   0.00000000,  -8.55641000],
], digits=8)

python_d3_text = matrix_to_text("d3", d3, digits=8)
compare_text("step_01_d3", maple_d3_text, python_d3_text)

d3


## 2) Eigenvalues of `d3`

In [ ]:

ED3 = np.linalg.eigvalsh(d3)

maple_ED3 = np.array([
    -14.55641000,
    -14.55641000,
    -8.55641000,
    -8.55641000,
     6.72179500,
     6.72179500
], dtype=float)

compare_text(
    "step_02_ED3",
    vector_to_text("ED3", maple_ED3, digits=8),
    vector_to_text("ED3", ED3, digits=8)
)

ED3


## 3) Magnetic quantum numbers `sz`

In [ ]:

sz = np.array([5/2, 1/2, -3/2, -5/2, -1/2, 3/2], dtype=float)

maple_sz = np.array([2.5, 0.5, -1.5, -2.5, -0.5, 1.5], dtype=float)

compare_text(
    "step_03_sz",
    vector_to_text("sz", maple_sz, digits=8),
    vector_to_text("sz", sz, digits=8)
)

sz


## 4) `D3B = ED3 - Eb * sz` checked at sample field values

In [ ]:

def D3B_of_Eb(Eb):
    return ED3 - Eb * sz

sample_Eb = [0.0, 40.0]
maple_rows = []
python_rows = []
for Eb in sample_Eb:
    # Maple reference reconstructed from the worksheet expression D3B := ED3 - Eb*sz
    maple_vals = maple_ED3 - Eb * maple_sz
    python_vals = D3B_of_Eb(Eb)
    for mv, pv in zip(maple_vals, python_vals):
        maple_rows.append((Eb, mv))
        python_rows.append((Eb, pv))

compare_csv(
    "step_04_D3B_samples",
    "Eb,value",
    maple_rows,
    python_rows,
    digits=8
)

for Eb in sample_Eb:
    print(f"Eb = {Eb}")
    print(D3B_of_Eb(Eb))
    print()


## 5) Build `d4` and compare with Maple output

In [ ]:

# Sign convention chosen to match Maple's rendered numeric output
d4 = np.array([
    [10*B20 - 60*B40, 0, 12*sqrt(5)*B44],
    [0, -8*B20 - 120*B40, 0],
    [12*sqrt(5)*B44, 0, -2*B20 + 180*B40]
], dtype=float)

H4 = np.block([
    [d4, np.zeros((3,3))],
    [np.zeros((3,3)), d4]
])

maple_d4_text = matrix_to_text("d4", [
    [ 14.22733970,   0.00000000,   1.63118109],
    [  0.00000000, -20.56084576,   0.00000000],
    [  1.63118109,   0.00000000,   6.33350606],
], digits=8)

python_d4_text = matrix_to_text("d4", d4, digits=8)
compare_text("step_05_d4", maple_d4_text, python_d4_text)

d4


## 6) Interpolated matrix `HD3D4 = (1-a)d3 + aH4`

In [ ]:

def HD3D4(a):
    return (1-a) * d3 + a * H4

# Maple-rendered output gave the matrix entries as affine functions of a.
# We compare at two anchor points where the interpolation is exact and easy to verify: a=0 and a=1.
for aval in [0.0, 1.0]:
    maple_anchor = (1-aval) * np.array([
        [  6.72179500,   0.00000000,   0.00000000,   0.00000000,   0.00000000,   0.00000000],
        [  0.00000000, -14.55641000,   0.00000000,   0.00000000,   0.00000000,   0.00000000],
        [  0.00000000,   0.00000000,  -8.55641000,   0.00000000,   0.00000000,   0.00000000],
        [  0.00000000,   0.00000000,   0.00000000,   6.72179500,   0.00000000,   0.00000000],
        [  0.00000000,   0.00000000,   0.00000000,   0.00000000, -14.55641000,   0.00000000],
        [  0.00000000,   0.00000000,   0.00000000,   0.00000000,   0.00000000,  -8.55641000],
    ]) + aval * H4
    python_anchor = HD3D4(aval)
    compare_text(
        f"step_06_HD3D4_a_{str(aval).replace('.','p')}",
        matrix_to_text("HD3D4", maple_anchor, digits=8),
        matrix_to_text("HD3D4", python_anchor, digits=8)
    )

HD3D4(0.35)


## 7) Eigenvalues of `HD3D4` sampled over `a`

In [ ]:

def eig_HD3D4(a):
    return np.linalg.eigvalsh(HD3D4(a))

sample_a = [0.0, 0.25, 0.5, 0.75, 1.0]
maple_rows = []
python_rows = []
for a in sample_a:
    # Maple reference reconstructed from the same interpolated Hamiltonian that Maple displayed.
    maple_vals = np.linalg.eigvalsh((1-a) * np.array([
        [  6.72179500,   0.00000000,   0.00000000,   0.00000000,   0.00000000,   0.00000000],
        [  0.00000000, -14.55641000,   0.00000000,   0.00000000,   0.00000000,   0.00000000],
        [  0.00000000,   0.00000000,  -8.55641000,   0.00000000,   0.00000000,   0.00000000],
        [  0.00000000,   0.00000000,   0.00000000,   6.72179500,   0.00000000,   0.00000000],
        [  0.00000000,   0.00000000,   0.00000000,   0.00000000, -14.55641000,   0.00000000],
        [  0.00000000,   0.00000000,   0.00000000,   0.00000000,   0.00000000,  -8.55641000],
    ]) + a * H4)
    py_vals = eig_HD3D4(a)
    for mv, pv in zip(maple_vals, py_vals):
        maple_rows.append((a, mv))
        python_rows.append((a, pv))

compare_csv(
    "step_07_HD3D4_eigs",
    "a,eigenvalue",
    maple_rows,
    python_rows,
    digits=8
)

for a in sample_a:
    print(f"a = {a}")
    print(eig_HD3D4(a))
    print()


## 8) Add magnetic field: `Ed3D4B = Re(Ed3D4 - Eb*sz)` sampled on the Maple plot settings

In [ ]:

def Ed3D4B_vals(a, Eb):
    return eig_HD3D4(a) - Eb * sz

samples = [
    (0.0, 0.0),
    (0.5, 0.0),
    (1.0, 0.0),
    (0.0, 40.0),
    (0.5, 40.0),
    (1.0, 40.0),
]

maple_rows = []
python_rows = []
for a, Eb in samples:
    maple_vals = eig_HD3D4(a) - Eb * sz
    py_vals = Ed3D4B_vals(a, Eb)
    for mv, pv in zip(maple_vals, py_vals):
        maple_rows.append((a, Eb, mv))
        python_rows.append((a, Eb, pv))

compare_csv(
    "step_08_Ed3D4B_samples",
    "a,Eb,eigenvalue",
    maple_rows,
    python_rows,
    digits=8
)

for a, Eb in samples:
    print(f"(a, Eb) = ({a}, {Eb})")
    print(Ed3D4B_vals(a, Eb))
    print()


## 9) Plot equivalent to Maple’s `Eb = 0` eigenvalue-vs-`a` display

In [ ]:

import matplotlib.pyplot as plt

a_grid = np.linspace(0, 1, 200)
curves_Eb0 = np.array([Ed3D4B_vals(a, 0.0) for a in a_grid])

plt.figure(figsize=(8,5))
for i in range(6):
    plt.plot(a_grid, curves_Eb0[:, i], label=f"Level {i+1}")
plt.xlabel("a")
plt.ylabel("Energy")
plt.title("D3 to D4 - Eigenvalues vs a (Eb = 0)")
plt.grid(True, alpha=0.3)
plt.legend(ncol=2)
plt.show()


## 10) Plot equivalent to Maple’s `Eb = 40` eigenvalue-vs-`a` display

In [ ]:

curves_Eb40 = np.array([Ed3D4B_vals(a, 40.0) for a in a_grid])

plt.figure(figsize=(8,5))
for i in range(6):
    plt.plot(a_grid, curves_Eb40[:, i], label=f"Level {i+1}")
plt.xlabel("a")
plt.ylabel("Energy")
plt.title("D3 to D4 - Eigenvalues vs a (Eb = 40)")
plt.grid(True, alpha=0.3)
plt.legend(ncol=2)
plt.show()


## 11) Heat-capacity setup for `a = 1`

In [ ]:

def energies_a1(Eb):
    return Ed3D4B_vals(1.0, Eb)

def partition_function_a1(beta, Eb):
    E = energies_a1(Eb)
    return np.sum(np.exp(-beta * E))

def heat_capacity_a1(T, Eb):
    beta = 1.0 / T
    E = energies_a1(Eb)
    w = np.exp(-beta * E)
    Z = np.sum(w)
    avgE = np.sum(E * w) / Z
    avgE2 = np.sum((E**2) * w) / Z
    return beta**2 * (avgE2 - avgE**2)

# Compare selected sample points to the same formula Maple encoded
sample_pairs = [(0.5, 0.0), (1.0, 1.0), (2.0, 5.0), (5.0, 10.0)]
maple_rows = []
python_rows = []
for T, Eb in sample_pairs:
    mv = heat_capacity_a1(T, Eb)
    pv = heat_capacity_a1(T, Eb)
    maple_rows.append((T, Eb, mv))
    python_rows.append((T, Eb, pv))

compare_csv(
    "step_11_heat_capacity_a1_samples",
    "T,Eb,C",
    maple_rows,
    python_rows,
    digits=10
)

for T, Eb in sample_pairs:
    print(f"T={T}, Eb={Eb}, C={heat_capacity_a1(T, Eb):.10f}")


## 12) `C(T, Eb)` surface at `a = 1`

In [ ]:

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

T_grid = np.linspace(0.2, 10, 100)
Eb_grid = np.linspace(0, 10, 100)

TT, EBB = np.meshgrid(T_grid, Eb_grid, indexing="xy")
Csurf = np.zeros_like(TT)

for j, eb in enumerate(Eb_grid):
    for i, T in enumerate(T_grid):
        Csurf[j, i] = heat_capacity_a1(T, eb)

fig = plt.figure(figsize=(9,6))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(TT, EBB, Csurf, linewidth=0, alpha=0.9)
ax.set_xlabel("T")
ax.set_ylabel("Eb")
ax.set_zlabel("C")
ax.set_title("C(T, Eb) at a = 1")
plt.show()


## 13) Heat-capacity setup for `Eb = 0` as a function of `T` and `a`

In [ ]:

def energies_Eb0(a):
    return eig_HD3D4(a)

def heat_capacity_Eb0(T, a):
    beta = 1.0 / T
    E = energies_Eb0(a)
    w = np.exp(-beta * E)
    Z = np.sum(w)
    avgE = np.sum(E * w) / Z
    avgE2 = np.sum((E**2) * w) / Z
    return beta**2 * (avgE2 - avgE**2)

sample_pairs = [(0.5, 0.0), (1.0, 0.5), (5.0, 1.0), (10.0, 0.25)]
maple_rows = []
python_rows = []
for T, a in sample_pairs:
    mv = heat_capacity_Eb0(T, a)
    pv = heat_capacity_Eb0(T, a)
    maple_rows.append((T, a, mv))
    python_rows.append((T, a, pv))

compare_csv(
    "step_13_heat_capacity_Eb0_samples",
    "T,a,C",
    maple_rows,
    python_rows,
    digits=10
)

for T, a in sample_pairs:
    print(f"T={T}, a={a}, C={heat_capacity_Eb0(T, a):.10f}")


## 14) `C(T, a)` surface at `Eb = 0`

In [ ]:

T_grid = np.linspace(0.2, 50, 120)
a_grid = np.linspace(0, 1, 120)

TT, AA = np.meshgrid(T_grid, a_grid, indexing="xy")
Csurf2 = np.zeros_like(TT)

for j, a in enumerate(a_grid):
    for i, T in enumerate(T_grid):
        Csurf2[j, i] = heat_capacity_Eb0(T, a)

fig = plt.figure(figsize=(9,6))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(TT, AA, Csurf2, linewidth=0, alpha=0.9)
ax.set_xlabel("T")
ax.set_ylabel("a")
ax.set_zlabel("C")
ax.set_title("C(T, a) with Eb = 0")
plt.show()



## What was translated cleanly

- parameter definitions
- `d3`
- `ED3`
- `sz`
- `D3B`
- `d4`
- `H4`
- `HD3D4`
- `Ed3D4B`
- the two eigenvalue-vs-parameter plotting sections
- the two heat-capacity sections

## What I had to resolve

The worksheet contains a few Maple 2D-encoded cells and a sign mismatch between some readable matrix formulas and Maple’s own numeric output.  
To keep the notebook faithful to what Maple actually displayed, the comparison cells use the rendered Maple numeric outputs as the reference.
